# Solución MLOps 01 — Google Colab

Este cuaderno es **autocontenido** para [Google Colab](https://colab.research.google.com/): instala dependencias, descarga el CSV de ejemplo desde el repositorio público y ejecuta el mismo pipeline mejorado (Random Forest + `BinaryEncoder` + `GridSearchCV`).

**Requisito:** conexión a Internet la primera vez (para `pip install` y descarga del archivo).

Si prefieres usar un CSV local, comenta la celda de descarga y sube `usuarios_promociones.csv` manualmente.


In [ ]:
# @title Entorno Colab (instala category_encoders si hace falta)
import subprocess
import sys

try:
    from category_encoders.binary import BinaryEncoder  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "category_encoders", "-q"])


In [ ]:
import pandas as pd

# CSV de referencia en el repositorio de la actividad (ajusta la URL si usas fork)
DATA_URL = "https://raw.githubusercontent.com/CarlosAMorcillo/MLOps_UdeM_Intro/main/usuarios_promociones.csv"

df = pd.read_csv(DATA_URL)
print(df.shape)
df.head()


## Actividad (solución): pipeline mejorado para promociones

Esta sección **continúa el notebook** con una versión más robusta del flujo de *machine learning* para predecir `dar_promocion`, cumpliendo la consigna de la actividad.

Incluye:

- **Feature engineering** (nuevas variables derivadas del comportamiento y del perfil).
- **Imputación alternativa** en números (`mean`, `median`, `constant` con valor `0`).
- **Codificación categórica** con `BinaryEncoder` (reduce dimensionalidad frente a *one-hot* completo en algunos casos).
- **Escalado** con `MinMaxScaler` (lleva cada variable numérica a un rango acotado, útil cuando queremos que todas contribuyan en una escala comparable antes del bosque).
- **Modelo** `RandomForestClassifier`.
- **Ajuste de hiperparámetros** con `GridSearchCV` (validación cruzada, optimizando F1).
- **Comparación experimental** de proporciones de *train/test* (80/20, 75/25 y 70/30) con estratificación.

> **Nota:** el `Pipeline` + `ColumnTransformer` mantiene el orden correcto: primero *train/test split*, luego `fit` solo sobre entrenamiento — así evitamos **data leakage** al imputar o escalar.


In [ ]:
# --- Imports adicionales para la solución de la actividad ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

from category_encoders.binary import BinaryEncoder

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


### Ingeniería de características (feature engineering)

A partir de una copia `df_mlops`, creamos variables sencillas interpretables:

| Feature | Idea |
|--------|------|
| `pages_per_minute` | proxies de ritmo de navegación usando sesiones, páginas por sesión y tiempo en sitio. |
| `estimated_total_value` | valor histórico aproximado: compras × ticket medio. |
| `estimated_abandoned_sessions` | sesiones recientes × tasa de abandono de carrito. |
| `is_recent_customer` | si lleva poco tiempo registrado (menos de 180 días). |
| `purchases_per_session` | intensidad de compra por sesión. |
| `minutes_per_page` | tiempo por “unidad de exposición” de contenido. |

Donde haga falta, usamos `.clip(lower=1)` para **evitar división por cero**. Los `NaN` que surjan se gestionan después con `SimpleImputer` dentro del *pipeline*.


In [ ]:
# --- Copia del dataframe y nuevas features ---
df_mlops = df.copy()

# Evitar división por cero en denominadores comunes
sessions_safe = df_mlops["sessions_last_30_days"].clip(lower=1)
time_safe = df_mlops["time_on_site_minutes"].clip(lower=1)
# “Volumen” de páginas vistas recientemente (proxy de actividad)
pages_activity = (df_mlops["sessions_last_30_days"] * df_mlops["pages_per_session"]).clip(lower=1)

df_mlops["pages_per_minute"] = (df_mlops["sessions_last_30_days"] * df_mlops["pages_per_session"]) / time_safe
df_mlops["estimated_total_value"] = df_mlops["total_purchases"] * df_mlops["avg_order_value"]
df_mlops["estimated_abandoned_sessions"] = df_mlops["sessions_last_30_days"] * df_mlops["cart_abandonment_rate"]
df_mlops["is_recent_customer"] = (df_mlops["days_since_registration"] < 180).astype(int)
df_mlops["purchases_per_session"] = df_mlops["total_purchases"] / sessions_safe
df_mlops["minutes_per_page"] = df_mlops["time_on_site_minutes"] / pages_activity

print("Columnas tras feature engineering:", df_mlops.shape[1])
display(df_mlops[["pages_per_minute", "estimated_total_value", "estimated_abandoned_sessions",
                  "is_recent_customer", "purchases_per_session", "minutes_per_page"]].describe())


### Diseño experimental: splits e imputación

Se prueban **9 combinaciones** (3 proporciones de test × 3 estrategias de imputación numérica).

En cada caso se entrena `GridSearchCV` con `cv=3`, `scoring="f1"` y un *grid* sobre hiperparámetros del bosque aleatorio. Así obtenemos una tabla ordenada para comparar configuraciones de forma objetiva.

**Importante:** `train_test_split(..., stratify=y_all)` conserva la proporción de la clase positiva en train y test.


In [ ]:
# --- Target, features y detección dinámica de tipos ---
# Excluimos identificador: no aporta patrón y encarecería el encoder
DROP_GROUP = ["dar_promocion", "user_id"]
X_all = df_mlops.drop(columns=DROP_GROUP)
y_all = df_mlops["dar_promocion"].astype(int)

numeric_cols = X_all.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_all.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numéricas:", len(numeric_cols), numeric_cols[:5], "...")
print("Categóricas:", categorical_cols)

# --- Grid de hiperparámetros del RandomForest ---
param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 5, 10],
    "classifier__min_samples_split": [2, 5],
    "classifier__min_samples_leaf": [1, 2],
}

SPLIT_CONFIGS = [(0.20, "80/20"), (0.25, "75/25"), (0.30, "70/30")]
NUM_STRATEGIES = [
    ("mean", {}),
    ("median", {}),
    ("constant", {"fill_value": 0}),
]

results_rows = []
fitted_grids = {}


def build_full_pipeline(num_strategy: str, num_kw: dict):
    """Construye Pipeline: preprocesamiento + RandomForest."""
    if num_strategy == "constant":
        num_imputer = SimpleImputer(strategy="constant", fill_value=num_kw.get("fill_value", 0))
    else:
        num_imputer = SimpleImputer(strategy=num_strategy)

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", num_imputer),
            ("scaler", MinMaxScaler()),
        ]
    )
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("encoder", BinaryEncoder()),
        ]
    )
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_cols),
            ("cat", categorical_pipeline, categorical_cols),
        ]
    )
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
        ]
    )


for test_size, split_label in SPLIT_CONFIGS:
    for num_strategy, num_kw in NUM_STRATEGIES:
        X_train, X_test, y_train, y_test = train_test_split(
            X_all,
            y_all,
            test_size=test_size,
            random_state=RANDOM_STATE,
            stratify=y_all,
        )
        pipeline = build_full_pipeline(num_strategy, num_kw)
        grid = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            cv=3,
            scoring="f1",
            n_jobs=-1,
            refit=True,
        )
        grid.fit(X_train, y_train)

        best_model = grid.best_estimator_
        y_pred = best_model.predict(X_test)
        y_proba = best_model.predict_proba(X_test)[:, 1]

        row = {
            "split": split_label,
            "imputation_strategy": num_strategy,
            "best_params": grid.best_params_,
            "cv_f1_best": grid.best_score_,
            "test_accuracy": accuracy_score(y_test, y_pred),
            "test_precision": precision_score(y_test, y_pred, zero_division=0),
            "test_recall": recall_score(y_test, y_pred, zero_division=0),
            "test_f1": f1_score(y_test, y_pred, zero_division=0),
            "test_roc_auc": roc_auc_score(y_test, y_proba),
        }
        results_rows.append(row)
        fitted_grids[(split_label, num_strategy)] = grid

results_df = pd.DataFrame(results_rows)
results_df["best_params_str"] = results_df["best_params"].apply(
    lambda d: ", ".join(f"{k}={v}" for k, v in sorted(d.items()))
)
results_df = results_df.sort_values(
    by=["test_f1", "test_roc_auc", "test_precision", "test_recall"],
    ascending=False,
).reset_index(drop=True)

print("Tabla de resultados (ordenada por test_f1 y criterios de desempate):")
display(
    results_df[
        [
            "split",
            "imputation_strategy",
            "cv_f1_best",
            "best_params_str",
            "test_accuracy",
            "test_precision",
            "test_recall",
            "test_f1",
            "test_roc_auc",
        ]
    ]
)


#### Resultados agregados

La tabla muestra, para cada experimento, el F1 medio en validación cruzada (referencia de `GridSearchCV`), los hiperparámetros elegidos y el desempeño **en el conjunto de prueba** (`accuracy`, `precision`, `recall`, `F1`, ROC-AUC).

La fila superior tras ordenar corresponde a la **configuración ganadora** según F1 en test (con criterios de desempate indicados en el código).


In [ ]:
# --- Mejor configuración según F1 en test (primera fila de la tabla ordenada) ---
winner = results_df.iloc[0]
print("Configuración ganadora:")
print(winner[["split", "imputation_strategy", "cv_f1_best", "test_f1", "test_roc_auc"]])

win_key = (winner["split"], winner["imputation_strategy"])
winner_grid = fitted_grids[win_key]

# Reconstruimos el split para evaluar con el mismo test reproducible
test_size_map = {"80/20": 0.20, "75/25": 0.25, "70/30": 0.30}
ts = test_size_map[winner["split"]]
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_all, y_all, test_size=ts, random_state=RANDOM_STATE, stratify=y_all
)

best_model = winner_grid.best_estimator_
# Ya está ajustado con refit=True sobre el train del experimento ganador (mismo split que abajo).
y_pred_w = best_model.predict(X_test_w)
y_proba_w = best_model.predict_proba(X_test_w)[:, 1]

print(f"\nROC-AUC (test, mejor modelo): {roc_auc_score(y_test_w, y_proba_w):.4f}\n")
print(classification_report(y_test_w, y_pred_w, digits=3))

cm = confusion_matrix(y_test_w, y_pred_w)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False)
ax.set_xlabel("Predicción")
ax.set_ylabel("Valor real")
ax.set_title("Matriz de confusión — mejor modelo (F1 en test)")
plt.tight_layout()
plt.show()


### Interpretación de métricas (caso promociones / `dar_promocion`)

En clasificación binaria para decidir si **dar o no una promoción**, cada métrica responde a una pregunta distinta:

- **F1-score** (media armónica de precisión y *recall*) equilibra los errores de los dos tipos sin asumir que uno es mucho más costoso que el otro. **Si el enunciado no define un costo de negocio dominante**, el F1 suele ser una métrica **centrada y razonable** para comparar modelos y para lo que optimizamos en `GridSearchCV` (`scoring="f1"`).

- **Accuracy** mide la fracción de aciertos globales. Con clases bastante balanceadas puede ser útil como lectura rápida, pero **puede ocultar desequilibrios** si una clase domina o si los costos de equivocarse en cada sentido son distintos.

- **Precisión** responde: *de los usuarios a los que el modelo marca para promoción, cuántos realmente debían recibirla*. Es clave cuando **las promociones son caras o el presupuesto es muy limitado**: prefieres dar menos promociones pero con mayor acierto para no “quemar” descuentos.

- **Recall** responde: *de todos los usuarios que merecían promoción, cuántos detectamos*. Importa cuando el objetivo es **no dejar fuera** a quienes sí conviene incentivar (maximizar cobertura del beneficio), aunque aceptes más falsos positivos.

- **ROC-AUC** resume la capacidad de **separar las dos clases** a distintos umbrales, útil cuando quieres comparar modelos de forma global sin fijar aún un umbral operativo concreto.

**Conclusión práctica:** al no especificarse en el enunciado un costo asimétrico (ni presupuesto ultra restrictivo ni mandato de máxima cobertura), **tomar F1 como métrica principal de selección** es una elección coherente; luego se puede afinar el umbral o la métrica según política comercial (por ejemplo, priorizar precisión con presupuesto ajustado o *recall* para campañas agresivas de captación).


### Cierre MLOps: buenas prácticas para llevar esto a un proyecto real

- **Persistencia del modelo:** guardar el *pipeline* completo (preprocesamiento + clasificador) con `joblib.dump(...)` para reproducir exactamente las transformaciones en producción.
- **Versionado:** etiquetar artefactos (dataset, pipeline, métricas) con versión o *commit* de Git; idealmente almacenar métricas en un registro (*MLflow*, etc.) aunque aquí no sea obligatorio.
- **Organización del código:** mover funciones reutilizables (carga de datos, *feature engineering*, construcción del *pipeline*) a módulos en `src/` y dejar el notebook para experimentación y narrativa.
- **Documentación:** mantener un `README` con objetivo, dependencias y cómo reproducir resultados.
- **Git:** usar **Conventional Commits** (por ejemplo `feat:`, `fix:`, `docs:`) para un historial legible y para automatizar *changelogs*.

Referencias: [Conventional Commits](https://www.conventionalcommits.org/en/v1.0.0/).
